# LushProtein — Solution 2 Standalone (Raw Data → Analysis)

**Fully self-contained notebook.** Requires only the five raw data folders at project root:

| Folder | Contents |
|--------|----------|
| `1.customer_transaction/` | Shopify order exports (`1_*.xlsx`) |
| `2.product_master/` | Product catalogue + unit costs |
| `3.Discounts/` | Discount code export |
| `4.Campaigns/` | Session / referrer data (reference) |
| `5.Recharge_data/` | Subscription (Recharge) exports |

**Pipeline inside this notebook:**
1. Install dependencies (first cell)
2. Load & merge raw Shopify transactions
3. Apply DQ + LP founder filters → finals cohort
4. Enrich with COGS from product master (`Cost per item`)
5. Reproduce Solution 2 metrics (4-layer recommendation engine)

**Run from:** project root 

In [1]:
# ── 0. Install dependencies (run this cell first) ───────────────────────────
import importlib.util
import subprocess
import sys

REQUIRED = [
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn",
    "openpyxl", "pyarrow",
]

missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All dependencies already installed.")


Installing: scikit-learn


In [2]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "1.customer_transaction").exists():
    alt = PROJECT_ROOT.parent
    if (alt / "1.customer_transaction").exists():
        PROJECT_ROOT = alt
    else:
        raise FileNotFoundError(
            "Open this notebook from the project root (folder containing 1.customer_transaction/)."
        )
os.chdir(PROJECT_ROOT)

RAW_ORDERS_DIR = PROJECT_ROOT / "1.customer_transaction"
STANDALONE_OUT = PROJECT_ROOT / "standalone_outputs"
STANDALONE_OUT.mkdir(exist_ok=True)

def _glob_one(folder: Path, pattern: str) -> Path:
    matches = sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern} in {folder}")
    return matches[0]

ORDER_FILES = sorted(RAW_ORDERS_DIR.glob("1_*.xlsx"))
PRODUCTS_FILE = _glob_one(PROJECT_ROOT / "2.product_master", "*.xlsx")
DISCOUNTS_FILE = _glob_one(PROJECT_ROOT / "3.Discounts", "*.csv")
CAMPAIGNS_FILE = _glob_one(PROJECT_ROOT / "4.Campaigns", "*.csv")
RECHARGE_DIR = PROJECT_ROOT / "5.Recharge_data"

EXCLUDE_HANDLE = "better-whey-protein-elite"
EXCLUDE_MONTHS = {7, 11}
ANALYSIS_START = pd.Timestamp("2022-01-01", tz="Asia/Singapore")
ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="UTC")
MARGIN_PROXY = 0.40
DECILE_CHART_ORDER = [f"D{i}" for i in range(10, 0, -1)]

FX_RATES_TO_SGD = {"SG": 1.0, "MY": 1.0 / 3.30, "HK": 1.0 / 6.10}
PRODUCT_MAP = {
    "lean-protein": "Lean Protein", "lean_protein": "Lean Protein",
    "clear-protein": "Clear Protein", "clear_protein": "Clear Protein",
    "collagen": "Collagen Glow", "soy-protein": "Soy Protein",
    "protein-bar": "Protein Bar", "shaker": "Accessories", "starter-kit": "Accessories",
}
MARKETPLACE_KEYWORDS = ["shopee", "lazada", "tokopedia", "redmart", "grab"]
CATEGORIES = [
    "Clear Protein", "Lean Protein", "Collagen Glow",
    "Accessories", "Soy Protein", "Other", "Unknown",
]

print("Project root:", PROJECT_ROOT)
print("Order files:", len(ORDER_FILES))
print("Products:", PRODUCTS_FILE.name)
print("Output dir:", STANDALONE_OUT)


Project root: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505
Order files: 7
Products: 2_1.products_master_20260505.xlsx
Output dir: c:\Users\adity\Documents\Aditya SMU\SMU Sem 5\Lush Protein SMU X\LushProtein_Project_Data_20260505\standalone_outputs


In [3]:
# ── Helper functions (mirrors EDA/00_config.py + aditya_findings/_shared.py) ─

def classify_product(handle) -> str:
    if pd.isna(handle):
        return "Unknown"
    h = str(handle).lower()
    for kw, label in PRODUCT_MAP.items():
        if kw in h:
            return label
    return "Other"


def classify_channel(row) -> str:
    tags = str(row.get("Tags", "") or "").lower()
    utm = str(row.get("Browser: UTM Source", "") or "").lower()
    name = str(row.get("Name", "") or "").lower()
    if any(k in tags for k in MARKETPLACE_KEYWORDS):
        return "Marketplace"
    if "subscription" in tags or "yotpo subscriptions" in tags or "lpsg" in name[:4]:
        return "Subscription"
    if utm in ("facebook", "instagram", "tiktok"):
        return "Paid Social"
    if utm in ("google", "bing"):
        return "Paid Search"
    if utm == "affiliate":
        return "Affiliate"
    if utm in ("shopify_email", "email", "klaviyo"):
        return "Email"
    return "Direct / Organic"


def store_prefix(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    n = str(name).upper().replace("#", "")
    if n.startswith("LPMY"):
        return "MY"
    if n.startswith("LPHK"):
        return "HK"
    if n.startswith("LPSG") or n.startswith("LP"):
        return "SG"
    return "Other"


def assign_decile(series: pd.Series) -> pd.Series:
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, q=10, labels=DECILE_CHART_ORDER)


def crm_tier(row) -> str:
    if row.get("is_top_both"):
        return "VIP"
    if row.get("is_top_profit") or row.get("profit_decile_true") == "D1":
        return "Profit_D1"
    if row.get("is_top_freq") or row.get("freq_decile_true") == "D1":
        return "Freq_D1"
    return "Standard"


def _safe_str(val, default="") -> str:
    """Convert cell values to str without boolean checks on pd.NA."""
    if val is None:
        return default
    try:
        if pd.isna(val):
            return default
    except (TypeError, ValueError):
        pass
    s = str(val).strip()
    if s in ("nan", "None", "<NA>", ""):
        return default
    return s


def sku_label(row) -> str:
    handle = _safe_str(row.get("Line: Product Handle"), "unknown")
    if handle == "unknown":
        handle = _safe_str(row.get("Line: SKU"), "unknown")
    variant = _safe_str(row.get("Line: Variant Title"), "")
    if "/" in variant:
        flavour = variant.split("/")[-1].strip()
    else:
        flavour = variant[:30] if variant else "default"
    return f"{handle}|{flavour}"[:80]


def clean_object_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].where(out[col].notna(), other=pd.NA)
        out[col] = out[col].apply(lambda x: str(x) if pd.notna(x) else pd.NA)
    return out


def build_cogs_map(products_df: pd.DataFrame) -> dict[str, float]:
    """COGS from product master Cost per item (Variant SKU key)."""
    cost_map: dict[str, float] = {}
    sku_col = "Variant SKU" if "Variant SKU" in products_df.columns else "SKU"
    cost_col = "Cost per item" if "Cost per item" in products_df.columns else None
    if cost_col is None:
        return cost_map
    for _, r in products_df[[sku_col, cost_col]].dropna(subset=[cost_col]).iterrows():
        cost_map[str(r[sku_col]).strip()] = float(r[cost_col])
    return cost_map


print("Helpers loaded.")


Helpers loaded.


## Part A — Load raw data

In [4]:
# ── A1. Shopify order transactions ────────────────────────────────────────────
assert ORDER_FILES, f"No 1_*.xlsx files in {RAW_ORDERS_DIR}"

raw_chunks = []
for f in ORDER_FILES:
    print(f"  {f.name} ...", end=" ")
    df = pd.read_excel(f, dtype={"ID": str, "Customer: ID": str})
    print(f"{len(df):,} rows")
    raw_chunks.append(df)

raw = pd.concat(raw_chunks, ignore_index=True)
raw["Processed At"] = pd.to_datetime(raw["Processed At"], utc=True, errors="coerce")
raw["order_date"] = raw["Processed At"].dt.tz_convert("Asia/Singapore").dt.normalize()
raw["store"] = raw["Name"].apply(store_prefix)

orders_cols = [
    "ID", "Name", "Tags", "order_date", "store", "Customer: ID", "Currency",
    "Price: Total", "Price: Total Discount", "Price: Total Shipping",
    "Payment: Status", "Order Fulfillment Status",
    "Shipping: Country", "Browser: UTM Source", "Browser: UTM Medium",
    "Line: Product Handle", "Line: Title", "Line: Variant Title", "Line: SKU",
    "Line: Price", "Line: Quantity",
]
existing_cols = [c for c in orders_cols if c in raw.columns]
orders_df = raw[raw["Top Row"] == 1][existing_cols].copy()
orders_df = orders_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
orders_df = orders_df.dropna(subset=["customer_id", "order_date"])
if "Payment: Status" in orders_df.columns:
    orders_df = orders_df[
        orders_df["Payment: Status"].isin(["paid", "partially_refunded"]) | orders_df["Payment: Status"].isna()
    ]
orders_df = orders_df[orders_df["Order Fulfillment Status"].fillna("") != "restocked"]

orders_df["_fx"] = orders_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Price: Total", "Price: Total Discount", "Price: Total Shipping", "Line: Price"]:
    if col in orders_df.columns:
        orders_df[col] = pd.to_numeric(orders_df[col], errors="coerce").fillna(0) * orders_df["_fx"]
orders_df.drop(columns=["_fx"], inplace=True)
orders_df["Currency"] = "SGD"
orders_df["channel"] = orders_df.apply(classify_channel, axis=1)
orders_df["product_category"] = orders_df["Line: Product Handle"].apply(classify_product)
orders_df["has_discount"] = orders_df["Price: Total Discount"].fillna(0) > 0
orders_df["is_subscription"] = orders_df["Tags"].fillna("").str.lower().str.contains("subscription|yotpo subscriptions")

line_cols = [
    "ID", "Customer: ID", "order_date", "store",
    "Line: Product Handle", "Line: Title", "Line: Variant Title",
    "Line: SKU", "Line: Quantity", "Line: Price", "Line: Discount", "Line: Total",
]
lines_df = raw[raw["Line: Type"] == "Line Item"][[c for c in line_cols if c in raw.columns]].copy()
lines_df = lines_df.rename(columns={"ID": "order_id", "Customer: ID": "customer_id"})
lines_df = lines_df.dropna(subset=["customer_id", "order_date"])
lines_df["product_category"] = lines_df["Line: Product Handle"].apply(classify_product)
lines_df["_fx"] = lines_df["store"].map(FX_RATES_TO_SGD).fillna(1.0)
for col in ["Line: Price", "Line: Discount", "Line: Total"]:
    if col in lines_df.columns:
        lines_df[col] = pd.to_numeric(lines_df[col], errors="coerce").fillna(0) * lines_df["_fx"]
lines_df.drop(columns=["_fx"], inplace=True)

cust_base = (
    orders_df.sort_values("order_date")
    .groupby("customer_id")
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        total_orders=("order_id", "count"),
        total_revenue=("Price: Total", "sum"),
        total_discount=("Price: Total Discount", "sum"),
        ever_subscribed=("is_subscription", "any"),
        ever_discounted=("has_discount", "any"),
        first_channel=("channel", "first"),
        first_product_cat=("product_category", "first"),
        first_store=("store", "first"),
    )
    .reset_index()
)
cust_base["cohort_month"] = cust_base["first_order_date"].dt.to_period("M")
second_orders = (
    orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    .rename(columns={"order_date": "second_order_date"})
)
cust_base = cust_base.merge(second_orders, on="customer_id", how="left")
cust_base["days_to_second"] = (cust_base["second_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["is_repeat"] = cust_base["total_orders"] >= 2
cust_base["lifespan_days"] = (cust_base["last_order_date"] - cust_base["first_order_date"]).dt.days
cust_base["recency_days"] = (ANALYSIS_DATE - cust_base["last_order_date"]).dt.days

orders_df = clean_object_cols(orders_df)
lines_df = clean_object_cols(lines_df)
cust_base = clean_object_cols(cust_base)

print(f"\nOrders: {len(orders_df):,} | Lines: {len(lines_df):,} | Customers: {len(cust_base):,}")
print(f"Date range: {orders_df['order_date'].min().date()} → {orders_df['order_date'].max().date()}")


  1_1.orders-2020_20260505.xlsx ... 12,201 rows
  1_2.orders-2021_20260505.xlsx ... 33,196 rows
  1_3.orders-2022_20260505.xlsx ... 18,563 rows
  1_4.orders-2023_20260505.xlsx ... 12,962 rows
  1_5.orders-2024_20260505.xlsx ... 26,373 rows
  1_6.orders-2025_20260505.xlsx ... 40,932 rows
  1_7.orders-2026_20260505.xlsx ... 9,601 rows

Orders: 27,350 | Lines: 50,963 | Customers: 13,780
Date range: 2020-01-01 → 2026-03-31


In [5]:
# ── A2. Ancillary raw tables ──────────────────────────────────────────────────
products_df = pd.read_excel(PRODUCTS_FILE)
discounts_df = pd.read_csv(DISCOUNTS_FILE, encoding="utf-8", encoding_errors="replace")
campaigns_df = pd.read_csv(CAMPAIGNS_FILE, encoding="utf-8", encoding_errors="replace")

rc_orders = pd.read_excel(_glob_one(RECHARGE_DIR, "*orders_combined*.xlsx"))
rc_checkout = pd.read_excel(_glob_one(RECHARGE_DIR, "*checkout*.xlsx"))
rc_reactivated = pd.read_excel(_glob_one(RECHARGE_DIR, "*reactivated*.xlsx"))
rc_churned = pd.read_excel(_glob_one(RECHARGE_DIR, "*churned*.xlsx"))
rc_recurring = pd.read_excel(_glob_one(RECHARGE_DIR, "*recurring*.xlsx"))

cost_map = build_cogs_map(products_df)
print("Products:", len(products_df), "| Discounts:", len(discounts_df), "| Campaigns:", len(campaigns_df))
print("Recharge tables:", len(rc_orders), len(rc_checkout), len(rc_reactivated), len(rc_churned), len(rc_recurring))
print("SKUs with COGS from product master:", len(cost_map))


Products: 167 | Discounts: 367 | Campaigns: 137033
Recharge tables: 1215 1094 50 526 650
SKUs with COGS from product master: 53


## Part B — Build finals cohort (DQ + LP filters)

Mirrors `EDA/13_build_finals_datasets.py`:
- **Layer 1:** DQ-02/03/04 order drops
- **Layer 2:** LP-F01/F02/F03/F04 customer flags
- **Layer 0:** 2022+ order window
- **Layer 3:** Drop Jul/Nov order months + elite SKU lines


In [6]:
def rebuild_customers(orders_df, lines_df, cust_seed, analysis_date=ANALYSIS_DATE):
    orders_df = orders_df.copy()
    orders_df["_rev_sgd"] = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
    agg = (
        orders_df.groupby("customer_id")
        .agg(total_orders=("order_id", "count"), total_revenue=("_rev_sgd", "sum"), last_order_date=("order_date", "max"))
        .reset_index()
    )
    orders_df.drop(columns=["_rev_sgd"], inplace=True, errors="ignore")

    active_ids = set(agg["customer_id"])
    cust = cust_seed[cust_seed["customer_id"].isin(active_ids)].copy()
    drop_cols = [
        "total_orders", "total_revenue", "first_order_date", "last_order_date", "is_repeat",
        "acq_year", "acq_month", "lifespan_days", "recency_days", "days_to_second", "second_order_date",
        "first_disc_depth", "first_disc_bin", "first_order_source", "first_order_pos",
        "exclude_elite_buyer", "exclude_51pct", "exclude_promo_month", "finals_eligible",
    ]
    cust = cust.drop(columns=[c for c in drop_cols if c in cust.columns])
    cust = cust.merge(agg, on="customer_id", how="inner")

    acq_cols = cust_seed[["customer_id", "first_order_date"]].drop_duplicates("customer_id")
    cust = cust.merge(acq_cols, on="customer_id", how="left")

    cust["is_repeat"] = cust["total_orders"] >= 2
    cust["acq_year"] = cust["first_order_date"].dt.year
    cust["acq_month"] = cust["first_order_date"].dt.month
    cust["lifespan_days"] = (cust["last_order_date"] - cust["first_order_date"]).dt.days
    cust["recency_days"] = (analysis_date - cust["last_order_date"]).dt.days

    first_ord = orders_df.sort_values("order_date").groupby("customer_id").first().reset_index()
    first_ord["first_rev"] = pd.to_numeric(first_ord["Price: Total"], errors="coerce").fillna(0)
    first_ord["first_disc"] = pd.to_numeric(first_ord["Price: Total Discount"], errors="coerce").fillna(0)
    first_ord["first_disc_depth"] = np.where(
        (first_ord["first_rev"] + first_ord["first_disc"]) > 0,
        first_ord["first_disc"] / (first_ord["first_rev"] + first_ord["first_disc"]), 0,
    )
    first_ord["first_disc_bin"] = pd.cut(
        first_ord["first_disc_depth"],
        bins=[-0.001, 0.001, 0.05, 0.10, 0.20, 0.30, 0.50, 1.01],
        labels=["0%", "1-5%", "6-10%", "11-20%", "21-30%", "31-50%", "51%+"],
    )
    if "Source" in first_ord.columns:
        first_ord["first_order_source"] = first_ord["Source"].fillna("unknown").str.lower()
    else:
        first_ord["first_order_source"] = "unknown"

    second_ord = orders_df.sort_values("order_date").groupby("customer_id", as_index=False).nth(1)[["customer_id", "order_date"]]
    second_ord = second_ord.rename(columns={"order_date": "second_order_date"})
    cust = cust.merge(first_ord[["customer_id", "first_disc_depth", "first_disc_bin", "first_order_source"]], on="customer_id", how="left")
    cust = cust.merge(second_ord, on="customer_id", how="left")
    cust["days_to_second"] = (cust["second_order_date"] - cust["first_order_date"]).dt.days
    cust["first_order_pos"] = cust["first_order_source"] == "pos"

    for col in ["first_channel", "ever_subscribed", "ever_discounted", "first_product_cat"]:
        if col not in cust.columns and col in cust_seed.columns:
            cust = cust.merge(cust_seed[["customer_id", col]].drop_duplicates("customer_id"), on="customer_id", how="left")

    elite_customers = set(
        lines_df[lines_df["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)]["customer_id"]
    )
    cust["exclude_elite_buyer"] = cust["customer_id"].isin(elite_customers)
    cust["exclude_51pct"] = cust["first_disc_bin"].astype(str) == "51%+"
    cust["exclude_promo_month"] = cust["acq_month"].isin(EXCLUDE_MONTHS)
    cust["finals_eligible"] = (
        (cust["first_order_date"] >= ANALYSIS_START)
        & ~cust["exclude_elite_buyer"]
        & ~cust["exclude_51pct"]
        & ~cust["exclude_promo_month"]
    )
    return cust


# Enrich orders with POS/web Source from raw files
src_chunks = []
for f in ORDER_FILES:
    if f.stat().st_size < 1000:
        continue
    src = pd.read_excel(f, usecols=["ID", "Top Row", "Source"], dtype=str, engine="openpyxl")
    src = src[src["Top Row"] == "1"].rename(columns={"ID": "order_id"})
    src_chunks.append(src[["order_id", "Source"]])
if src_chunks:
    src_df = pd.concat(src_chunks, ignore_index=True).drop_duplicates("order_id")
    src_df["order_id"] = src_df["order_id"].astype(str)
    orders_df["order_id"] = orders_df["order_id"].astype(str)
    orders_df = orders_df.merge(src_df, on="order_id", how="left")

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"], utc=True)
lines_df["order_date"] = pd.to_datetime(lines_df["order_date"], utc=True)
orders_df["order_id"] = orders_df["order_id"].astype(str)
lines_df["order_id"] = lines_df["order_id"].astype(str)
cust_base["first_order_date"] = pd.to_datetime(cust_base["first_order_date"], utc=True)

# Layer 1 — DQ
_rev = pd.to_numeric(orders_df["Price: Total"], errors="coerce").fillna(0)
_disc = pd.to_numeric(orders_df["Price: Total Discount"], errors="coerce").fillna(0)
_dq02 = (_rev == 0) & (_disc == 0)
_dq03 = (_rev == 0) & (_disc > 0)
_dq04 = orders_df["Tags"].fillna("").str.lower().str.contains("wholesale") | (_rev > 5000)
orders_dq = orders_df[~(_dq02 | _dq03 | _dq04)].copy()
lines_dq = lines_df[lines_df["order_id"].isin(set(orders_dq["order_id"]))].copy()
cust_dq = rebuild_customers(orders_dq, lines_dq, cust_base)

# Layer 2
finals_ids = set(cust_dq[cust_dq["finals_eligible"]]["customer_id"])
orders_l2 = orders_dq[orders_dq["customer_id"].isin(finals_ids)].copy()
lines_l2 = lines_dq[lines_dq["customer_id"].isin(finals_ids)].copy()
cust_l2 = cust_dq[cust_dq["finals_eligible"]].copy()

# Layer 0
orders_l2 = orders_l2[orders_l2["order_date"] >= ANALYSIS_START].copy()
lines_l2 = lines_l2[lines_l2["order_id"].isin(set(orders_l2["order_id"]))].copy()

# Layer 3
_jul_nov = orders_l2["order_date"].dt.month.isin(EXCLUDE_MONTHS)
orders_finals = orders_l2[~_jul_nov].copy()
lines_finals = lines_l2[
    lines_l2["order_id"].isin(set(orders_finals["order_id"]))
    & ~lines_l2["Line: Product Handle"].fillna("").str.contains(EXCLUDE_HANDLE, case=False)
].copy()

customers = rebuild_customers(orders_finals, lines_finals, cust_l2)
customers = customers[customers["customer_id"].isin(finals_ids)].copy()
customers["finals_eligible"] = True
orders = orders_finals.copy()
lines = lines_finals.copy()

print("Finals cohort:")
print(f"  Orders:    {len(orders):,}")
print(f"  Lines:     {len(lines):,}")
print(f"  Customers: {len(customers):,}")
print(f"  Finals-eligible flag: {customers['finals_eligible'].sum():,}")


Finals cohort:
  Orders:    8,955
  Lines:     14,448
  Customers: 5,694
  Finals-eligible flag: 5,694


## Part C — Margin enrichment (COGS from product master)

In [7]:
# Attach unit costs and gross profit to line items
lines["sku_key"] = lines["Line: SKU"].astype(str).str.strip()
lines["unit_cost"] = lines["sku_key"].map(cost_map)
lines["line_rev"] = pd.to_numeric(lines["Line: Total"], errors="coerce").fillna(0)
lines["qty"] = pd.to_numeric(lines["Line: Quantity"], errors="coerce").fillna(1).clip(lower=1)
lines["cogs"] = lines["unit_cost"] * lines["qty"]
lines["gross_profit"] = np.where(lines["unit_cost"].notna(), lines["line_rev"] - lines["cogs"], lines["line_rev"] * MARGIN_PROXY)
lines["has_cogs"] = lines["unit_cost"].notna()
lines["margin_pct"] = np.where(lines["line_rev"] > 0, lines["gross_profit"] / lines["line_rev"], np.nan)

order_gp = lines.groupby("order_id").agg(
    order_gp=("gross_profit", "sum"), order_cogs=("cogs", "sum"), order_rev=("line_rev", "sum"),
    n_categories=("product_category", "nunique"),
).reset_index()
order_gp["order_margin_pct"] = np.where(order_gp["order_rev"] > 0, order_gp["order_gp"] / order_gp["order_rev"], np.nan)
orders = orders.merge(order_gp, on="order_id", how="left")
orders["order_gp"] = orders["order_gp"].fillna(pd.to_numeric(orders["Price: Total"], errors="coerce").fillna(0) * MARGIN_PROXY)

cat_ever = lines.groupby("customer_id")["product_category"].nunique().reset_index(name="n_categories_ever")
cust_rev = orders.groupby("customer_id").agg(
    finals_revenue=("Price: Total", lambda s: pd.to_numeric(s, errors="coerce").sum()),
    finals_orders=("order_id", "count"),
    true_gross_profit=("order_gp", "sum"),
).reset_index()
customers = customers.merge(cat_ever, on="customer_id", how="left").merge(cust_rev, on="customer_id", how="left")
customers["true_gross_profit"] = customers["true_gross_profit"].fillna(customers["total_revenue"] * MARGIN_PROXY)
customers["n_categories_ever"] = customers["n_categories_ever"].fillna(1).astype(int)

pool = customers[customers["finals_eligible"]].copy()
pool["profit_decile_true"] = assign_decile(pool["true_gross_profit"])
pool["freq_decile_true"] = assign_decile(pool["finals_orders"].fillna(pool["total_orders"]))
pool["is_top_profit"] = pool["profit_decile_true"] == "D1"
pool["is_top_freq"] = pool["freq_decile_true"] == "D1"
pool["is_top_both"] = pool["is_top_profit"] & pool["is_top_freq"]
pool["crm_tier"] = pool.apply(crm_tier, axis=1)
customers = customers.merge(
    pool[["customer_id", "profit_decile_true", "freq_decile_true", "is_top_profit", "is_top_freq", "is_top_both", "crm_tier"]],
    on="customer_id", how="left",
)

# Save standalone parquet cache (optional — for inspection)
for name, df in [("orders", orders), ("lines", lines), ("customers", customers)]:
    df.to_parquet(STANDALONE_OUT / f"{name}.parquet", index=False)

manifest = {
    "source": "solution2_standalone_from_raw.ipynb",
    "row_counts": {"orders": len(orders), "lines": len(lines), "customers": len(customers)},
    "pct_lines_with_cogs": round(lines["has_cogs"].mean() * 100, 1),
}
(STANDALONE_OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


{
  "source": "solution2_standalone_from_raw.ipynb",
  "row_counts": {
    "orders": 8955,
    "lines": 14448,
    "customers": 5694
  },
  "pct_lines_with_cogs": 21.0
}


## Part D — Solution 2: Problem statement

In [8]:
fc = customers[customers["finals_eligible"] == True].copy()

one_and_done_pct = (fc["finals_orders"] == 1).mean() * 100
single_cat_pct = (fc["n_categories_ever"] == 1).mean() * 100
repeat_pct = (fc["finals_orders"] > 1).mean() * 100

print("=== PROBLEM STATEMENT ===")
print(f"Finals customers: {len(fc):,}")
print(f"One-and-done: {(fc['finals_orders'] == 1).sum():,} ({one_and_done_pct:.1f}%)")
print(f"Repeat customers: {(fc['finals_orders'] > 1).sum():,} ({repeat_pct:.1f}%)")
print(f"Single-category: {(fc['n_categories_ever'] == 1).sum():,} ({single_cat_pct:.1f}%)")

cat_ladder = (
    fc.groupby("n_categories_ever")
    .agg(n=("customer_id", "count"), avg_gp=("true_gross_profit", "mean"),
         repeat_rate=("finals_orders", lambda x: (x > 1).mean()), avg_orders=("finals_orders", "mean"))
    .round(3)
)
display(cat_ladder)

sub = fc[fc["ever_subscribed"]]
nonsub = fc[~fc["ever_subscribed"]]
print(f"\nSubscribers repeat: {(sub['finals_orders'] > 1).mean()*100:.1f}% (n={len(sub):,})")
print(f"Non-subscribers repeat: {(nonsub['finals_orders'] > 1).mean()*100:.1f}% (n={len(nonsub):,})")


=== PROBLEM STATEMENT ===
Finals customers: 5,694
One-and-done: 4,402 (77.3%)
Repeat customers: 1,292 (22.7%)
Single-category: 3,681 (64.6%)


,n,avg_gp,repeat_rate,avg_orders
n_categories_ever,,,,
1,3681,52.085,0.133,1.214
2,1411,70.598,0.300,1.638
3,445,118.399,0.555,2.326
4,123,191.822,0.821,3.911
5,31,194.911,0.839,4.129
6,2,237.100,1.000,3.500
7,1,18377.900,1.000,524.000



Subscribers repeat: 61.9% (n=713)
Non-subscribers repeat: 17.1% (n=4,981)


## Part E — Layer 1: Category co-purchase (cold start rules)

In [9]:
def build_co_purchase_matrix(line_df):
    cust_cats = line_df.groupby("customer_id")["product_category"].apply(set).reset_index()
    matrix = pd.DataFrame(index=CATEGORIES, columns=CATEGORIES, dtype=float)
    for row_cat in CATEGORIES:
        buyers = cust_cats[cust_cats["product_category"].apply(lambda s: row_cat in s)]
        n = len(buyers)
        if n == 0:
            continue
        for col_cat in CATEGORIES:
            if row_cat == col_cat:
                matrix.loc[row_cat, col_cat] = 100.0
            else:
                also = buyers["product_category"].apply(lambda s: col_cat in s).sum()
                matrix.loc[row_cat, col_cat] = round(also / n * 100, 1)
    return matrix

co_all = build_co_purchase_matrix(lines)
display(co_all.loc[["Clear Protein", "Lean Protein", "Collagen Glow"], ["Clear Protein", "Lean Protein", "Collagen Glow", "Accessories"]])
print(f"Clear → Lean: {co_all.loc['Clear Protein', 'Lean Protein']:.1f}%")
print(f"Lean → Clear: {co_all.loc['Lean Protein', 'Clear Protein']:.1f}%")


,Clear Protein,Lean Protein,Collagen Glow,Accessories
Clear Protein,100.0,25.0,7.6,23.6
Lean Protein,28.9,100.0,7.8,29.2
Collagen Glow,23.8,21.1,100.0,16.7


Clear → Lean: 25.0%
Lean → Clear: 28.9%


## Part F — Layer 2: Market basket analysis

In [10]:
lines_mba = lines.copy()
lines_mba["sku_display"] = lines_mba.apply(sku_label, axis=1)
baskets = lines_mba.groupby("order_id")["sku_display"].apply(lambda s: sorted(set(s))).reset_index()
n_orders = len(baskets)
item_counts = lines_mba.groupby("sku_display")["order_id"].nunique()
pair_counts = {}
for items in baskets["sku_display"]:
    if len(items) < 2:
        continue
    for a, b in combinations(items, 2):
        key = (a, b) if a < b else (b, a)
        pair_counts[key] = pair_counts.get(key, 0) + 1

MIN_SUPPORT, MIN_CONFIDENCE = 0.005, 0.05
rules = []
for (a, b), cnt in pair_counts.items():
    support = cnt / n_orders
    if support < MIN_SUPPORT:
        continue
    for ant, cons, conf in [(a, b, cnt / item_counts.get(a, 1)), (b, a, cnt / item_counts.get(b, 1))]:
        if conf >= MIN_CONFIDENCE:
            rules.append({"antecedent": ant, "consequent": cons, "support": round(support, 4), "confidence": round(conf, 4)})
rules_df = pd.DataFrame(rules).sort_values(["confidence", "support"], ascending=False).drop_duplicates(["antecedent", "consequent"])
print(f"Rules: {len(rules_df):,} | Top rule confidence: {rules_df.iloc[0]['confidence']:.2f}")
display(rules_df.head(10))


Rules: 46 | Top rule confidence: 0.93


,antecedent,consequent,support,confidence
22,0724999810463|Thai Milk Tea,lushprotein-clear-shaker|White,0.0161,0.9290
21,0724999810470|Taro,0724999810463|Thai Milk Tea,0.0137,0.9179
24,0724999810470|Taro,lushprotein-clear-shaker|White,0.0135,0.9030
20,0724999810463|Thai Milk Tea,0724999810470|Taro,0.0137,0.7935
29,clear-protein-25g-single-sachet|White Grape,clear-protein-25g-single-sachet|Peach,0.0170,0.7136
26,lushprotein-lean-protein-40g-single-serve|Taro,lushprotein-lean-protein-40g-single-serve|Thai...,0.0170,0.6786
28,clear-protein-25g-single-sachet|Peach,clear-protein-25g-single-sachet|White Grape,0.0170,0.6580
27,lushprotein-lean-protein-40g-single-serve|Thai...,lushprotein-lean-protein-40g-single-serve|Taro,0.0170,0.6080
10,lean-protein|Taro,lean-protein|Thai Milk Tea,0.0286,0.3590
1,clear-protein|White Grape,clear-protein|Peach,0.0372,0.3561


## Part G — Layer 3: Timed cross-sell & sample schedule

In [11]:
REORDER_MEDIAN_DAYS = {"Clear Protein": 54, "Lean Protein": 35, "Collagen Glow": 42, "Accessories": 35, "Soy Protein": 84, "Other": 50, "Unknown": 50}
CROSS_CATEGORY_MAP = {
    "Clear Protein": {"cross": "Lean Protein", "sample": "Collagen Glow", "email_day": 14},
    "Lean Protein": {"cross": "Clear Protein", "sample": "Collagen Glow", "email_day": 14},
    "Collagen Glow": {"cross": "Clear Protein", "sample": "Lean Protein", "email_day": 21},
    "Accessories": {"cross": "Lean Protein", "sample": "Clear Protein", "email_day": 7},
    "Soy Protein": {"cross": "Clear Protein", "sample": "Lean Protein", "email_day": 14},
    "Other": {"cross": "Clear Protein", "sample": "Lean Protein", "email_day": 14},
    "Unknown": {"cross": "Clear Protein", "sample": "Lean Protein", "email_day": 14},
}
timing_rows = []
for cat, spec in CROSS_CATEGORY_MAP.items():
    reorder = REORDER_MEDIAN_DAYS.get(cat, 50)
    email_day = spec["email_day"]
    sample_day = max(email_day + 7, reorder - 10)
    timing_rows.append({
        "first_product_category": cat,
        "cross_sell_category": spec["cross"],
        "sample_category": spec["sample"],
        "email_day": email_day,
        "physical_sample_day": sample_day,
        "median_reorder_days": reorder,
    })
timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(STANDALONE_OUT / "cross_sell_timing_and_samples.csv", index=False)
display(timing_df)


,first_product_category,cross_sell_category,sample_category,email_day,physical_sample_day,median_reorder_days
0,Clear Protein,Lean Protein,Collagen Glow,14,44,54
1,Lean Protein,Clear Protein,Collagen Glow,14,25,35
2,Collagen Glow,Clear Protein,Lean Protein,21,32,42
3,Accessories,Lean Protein,Clear Protein,7,25,35
4,Soy Protein,Clear Protein,Lean Protein,14,74,84
5,Other,Clear Protein,Lean Protein,14,40,50
6,Unknown,Clear Protein,Lean Protein,14,40,50


## Part H — Layer 4: Item-item collaborative filtering

In [12]:
lines_cf = lines.copy()
lines_cf["sku_display"] = lines_cf.apply(sku_label, axis=1)
cust_sku = (lines_cf.groupby(["customer_id", "sku_display"]).size().unstack(fill_value=0) > 0).astype(int)
keep = cust_sku.sum(axis=0)[cust_sku.sum(axis=0) >= 10].index.tolist()
item_sim = pd.DataFrame(cosine_similarity(cust_sku[keep].T), index=keep, columns=keep)

hero = next((s for s in ["clear-protein|Peach", "clear-protein|White Grape"] if s in item_sim.index), keep[0])
sims = item_sim.loc[hero].drop(hero, errors="ignore").sort_values(ascending=False).head(5)
print(f"Top CF neighbours for {hero}:")
for sku, score in sims.items():
    print(f"  {sku}: {score:.3f}")


Top CF neighbours for clear-protein|Peach:
  clear-protein|White Grape: 0.411
  lushprotein-clear-shaker|White: 0.216
  lean-protein|Thai Milk Tea: 0.164
  0724999808361|default: 0.142
  lushprotein-clear-shaker|default: 0.129


---

**Reproducibility checklist**
- [ ] Cell 0 installs all dependencies without manual pip
- [ ] Part A loads all 5 raw data folders
- [ ] Part B finals counts match expected (~8,955 orders / ~5,694 customers)
- [ ] Part C attaches COGS from product master
- [ ] Parts D–H reproduce Solution 2 presentation metrics

Outputs cached in `standalone_outputs/` (parquet + timing CSV + manifest).
